# 01 — Clustering des nœuds du graphe financier

Ce notebook explore et valide le clustering HDBSCAN des comptes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, '../src')

from load_data import load_features, load_edges
from preprocess import preprocess
from reduce_dimension import run_reduction
from clustering import run_clustering

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
df_features = load_features()
print(f'Nœuds : {len(df_features):,}')
print(f'Colonnes : {list(df_features.columns)}')

In [ ]:
X_scaled, cols, scaler = preprocess(df_features)
print(f'X_scaled shape : {X_scaled.shape}')

In [ ]:
# Réduction 3D
df_3d = run_reduction(X_scaled, df_features['node_id'])
df_3d.head()

In [ ]:
# Clustering
df_clusters = run_clustering(X_scaled, df_features['node_id'], df_features)
print(df_clusters['cluster_label'].value_counts())

In [ ]:
# Visualisation 2D des clusters (projection x,y)
merged = df_3d.merge(df_clusters[['node_id', 'cluster_label']], on='node_id')
merged = merged.merge(df_features[['node_id', 'is_fraud_node']], on='node_id')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Par cluster
for label in sorted(merged['cluster_label'].unique()):
    sub = merged[merged['cluster_label'] == label]
    color = 'gray' if label == -1 else None
    axes[0].scatter(sub['x'], sub['y'], s=3, alpha=0.5, label=f'C{label}', c=color)
axes[0].set_title('Clusters HDBSCAN (projection x,y)')
axes[0].legend(markerscale=4, loc='upper right')

# Par fraude
for fraud, color, label in [(0, 'steelblue', 'Normal'), (1, 'red', 'Fraude')]:
    sub = merged[merged['is_fraud_node'] == fraud]
    axes[1].scatter(sub['x'], sub['y'], s=3, alpha=0.5, c=color, label=label)
axes[1].set_title('Nœuds frauduleux vs normaux')
axes[1].legend(markerscale=4)

plt.tight_layout()
plt.show()

In [ ]:
# Stats par cluster
cluster_stats = df_clusters.groupby('cluster_label').agg(
    size=('node_id', 'count'),
    fraud_rate=('fraud_node_rate', 'mean'),
    avg_amount=('avg_total_amount', 'mean'),
).reset_index()
print(cluster_stats.sort_values('fraud_rate', ascending=False).to_string())